# VQA / OCR Benchmark

Benchmark frontier vision models on visual question answering and OCR tasks.
Compares accuracy, cost, and speed across providers and effort levels.

In [ ]:
!pip install -qU vlm-exam

## Configure API Keys

Add the following keys to your Colab secrets:
- `ANTHROPIC_API_KEY` from [console.anthropic.com](https://console.anthropic.com)
- `GOOGLE_API_KEY` from [aistudio.google.dev](https://aistudio.google.dev/apikey)
- `OPENAI_API_KEY` from [platform.openai.com](https://platform.openai.com/api-keys)

In [ ]:
import os

from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Download dataset

In [ ]:
from roboflow import Roboflow

rf = Roboflow()
project = rf.workspace("roboflow-jvuqo").project("roboflow_playground_vqa")
version = project.version(4)
dataset = version.download("jsonl")

DATA_DIR = dataset.location + "/train"

## Run benchmark

In [ ]:
from vlm_exam import create_provider, create_task, load_config, run_benchmark

config = load_config()
task = create_task("vqa")
samples = task.load_samples(DATA_DIR)

MODELS = ["claude-fable-5", "claude-sonnet-5", "gemini-3.5-flash", "gpt-5.5"]
EFFORT_LEVELS = ["low", "high"]

all_results = {}

for model_id in MODELS:
    for effort in EFFORT_LEVELS:
        model_config = config.models[model_id]
        provider = create_provider(model_config.provider, model=model_id)
        result = run_benchmark(
            task=task,
            provider=provider,
            samples=samples,
            effort=effort,
            task_name="vqa",
        )
        all_results[(model_id, effort)] = result

## Results summary

In [ ]:
print(f"{'Model':<25} {'Effort':>6} {'Correct':>8} {'Total':>6} {'Accuracy':>9}")
print("-" * 60)

for model_id in MODELS:
    for effort in EFFORT_LEVELS:
        result = all_results[(model_id, effort)]
        correct = sum(s.correct for s in result.samples)
        total = len(result.samples)
        accuracy = correct / total * 100
        print(f"{model_id:<25} {effort:>6} {correct:>8} {total:>6} {accuracy:>8.1f}%")

## Accuracy charts

In [ ]:
from vlm_exam.visualization import (
    plot_accuracy_chart,
    plot_combined_metrics_chart,
    plot_dual_effort_chart,
    plot_failure_card,
    plot_success_card,
)

for effort in EFFORT_LEVELS:
    accuracy = {}
    for model_id in MODELS:
        result = all_results[(model_id, effort)]
        accuracy[model_id] = sum(s.correct for s in result.samples) / len(result.samples) * 100

    figure = plot_accuracy_chart(accuracy, config, f"VQA / OCR Benchmark \u2014 {effort.title()} Effort")
    figure.show()

## Efficiency charts

In [ ]:
tokens_high, tokens_low = {}, {}
cost_high, cost_low = {}, {}
time_high, time_low = {}, {}

for model_id in MODELS:
    for effort in EFFORT_LEVELS:
        result = all_results[(model_id, effort)]
        pricing = config.models[model_id].pricing
        count = len(result.samples)

        average_input = sum(s.input_tokens for s in result.samples) / count
        average_output = sum(s.output_tokens for s in result.samples) / count
        average_cost = (
            (average_input / 1_000_000) * pricing.input_per_million_tokens
            + (average_output / 1_000_000) * pricing.output_per_million_tokens
        )
        timed = [s.elapsed_seconds for s in result.samples if s.elapsed_seconds is not None]
        average_time = sum(timed) / len(timed) if timed else 0.0

        target_tokens = tokens_high if effort == "high" else tokens_low
        target_cost = cost_high if effort == "high" else cost_low
        target_time = time_high if effort == "high" else time_low

        target_tokens[model_id] = average_input + average_output
        target_cost[model_id] = average_cost
        target_time[model_id] = average_time

plot_dual_effort_chart(
    tokens_high, tokens_low, config,
    "Avg Tokens per Image", lambda v: f"{v:,.0f}", sort_ascending=True,
).show()

plot_dual_effort_chart(
    cost_high, cost_low, config,
    "Avg Cost per Image", lambda v: f"${v:.5f}", sort_ascending=True,
).show()

plot_dual_effort_chart(
    time_high, time_low, config,
    "Avg Inference Time per Image", lambda v: f"{v:.1f}s", sort_ascending=True,
).show()

plot_combined_metrics_chart(
    tokens_high, tokens_low,
    cost_high, cost_low,
    time_high, time_low,
    config,
).show()

## Failure analysis

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

for model_id in MODELS:
    for effort in EFFORT_LEVELS:
        result = all_results[(model_id, effort)]
        failures = [s for s in result.samples if not s.correct]

        print(f"\n{'=' * 60}")
        print(f"  {model_id} ({effort}) \u2014 Failures: {len(failures)}/{len(result.samples)}")
        print(f"{'=' * 60}\n")

        for sample in failures:
            image = Image.open(f"{DATA_DIR}/{sample.image}").convert("RGB")
            question = sample.metadata.get("question", "")
            figure = plot_failure_card(
                image=image,
                question=question,
                expected=sample.expected,
                predicted=sample.predicted,
                model_id=model_id,
                config=config,
            )
            plt.show()
            plt.close(figure)